In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from typing import Annotated
from operator import add

graph_builder = StateGraph(Annotated[int, add])

graph_builder.add_node("get_one", lambda x: 1)
graph_builder.set_entry_point("get_one")
graph_builder.set_finish_point("get_one")

graph = graph_builder.compile(checkpointer=InMemorySaver())

result1 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"첫 번째 호출 결과: {result1}\n")
result2 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"두 번째 호출 결과: {result2}\n")
result3 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"세 번째 호출 결과: {result3}")

첫 번째 호출 결과: 1

두 번째 호출 결과: 2

세 번째 호출 결과: 3


In [7]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o")
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.set_entry_point("chatbot")
graph_builder.set_finish_point("chatbot")
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [8]:
graph.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! 제 이름은 공원나연입니다."}]},
    {"configurable": {"thread_id": "1"}},
)

{'messages': [HumanMessage(content='안녕하세요! 제 이름은 공원나연입니다.', additional_kwargs={}, response_metadata={}, id='b61044d6-b090-45de-85ae-4174a43351cc'),
  AIMessage(content='안녕하세요, 나연님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 19, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_b8176c105e', 'id': 'chatcmpl-EPnOk6QSIbecKV0FtGpREAfjDGlIj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b964-e899-78e3-bbbc-116223ad6c49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_to

In [9]:
graph.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐라고요?"}]},
    {"configurable": {"thread_id": "1"}},
)

{'messages': [HumanMessage(content='안녕하세요! 제 이름은 공원나연입니다.', additional_kwargs={}, response_metadata={}, id='b61044d6-b090-45de-85ae-4174a43351cc'),
  AIMessage(content='안녕하세요, 나연님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 19, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_b8176c105e', 'id': 'chatcmpl-EPnOk6QSIbecKV0FtGpREAfjDGlIj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b964-e899-78e3-bbbc-116223ad6c49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_to

In [4]:
from langchain_core.messages.utils import (
    trim_messages,
    count_tokens_approximately,
)
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("openai:gpt-4o")

def call_model(state: MessagesState):
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=128,
        start_on="human",
        end_on=("human", "tool"),
    )
    response = model.invoke(messages)
    return {"messages": [response]}

graph_builder = StateGraph(MessagesState)
graph_builder.add_node(call_model)
graph_builder.add_edge(START, "call_model")

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [5]:
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "안녕하세요, 저는 AI를 공부하는 학생입니다."}, config)
graph.invoke({"messages": "앞으로 AI를 사용할 때 주의해야 할 점을 조사하고 있어요."}, config)
graph.invoke({"messages": "또 어떤 주제에 대해 공부하면 좋을까요?"}, config)
final_response = graph.invoke({"messages": "제가 뭘 공부하고 있나요?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

죄송하지만, 현재 어떤 주제를 공부하고 계신지에 대한 정보를 가지고 있지 않습니다. 질문해주시면 학습하고 계신 주제에 대한 도움이 되는 자료나 정보를 제공해드릴 수 있습니다. 어떤 내용을 공부 중이신가요?


In [8]:
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

class State(MessagesState):
    summary: str

model = init_chat_model("openai:gpt-4o")

def summarize_conversation(state: State):
    summary = state.get("summary", "")

    if summary:
        summary_message = (
            f"지금까지 대화 요약: {summary}\n\n"
            "위의 새로운 메시지를 고려하여 요약을 확장해주세요:"
        )
    else:
        summary_message = "위 대화 내용을 요약해주세요:"

    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)

    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

In [9]:
def call_model(state: State):
    summary = state.get("summary", "")
    if summary:
        system_message = f"이전 대화 요약: {summary}"
        messages = [{"role": "system", "content": system_message}] + state["messages"]
    else:
        messages = state["messages"]

    response = model.invoke(messages)
    return {"messages": [response]}

def should_continue(state: State):
    mesages = state["messages"]
    if len(mesages) > 6:
        return "summarize_conversation"
    return "model"

graph_builder = StateGraph(State)
graph_builder.add_node("call_model", call_model)
graph_builder.add_node("summarize_conversation", summarize_conversation)

graph_builder.add_edge("summarize_conversation", "call_model")
graph_builder.add_conditional_edges(
    START,
    should_continue,
    {
        "model": "call_model",
        "summarize_conversation": "summarize_conversation",
    }
)
graph_builder.add_edge("call_model", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [10]:
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "안녕하세요, 저는 AI를 공부하는 학생입니다."}, config)
graph.invoke({"messages": "앞으로 AI를 사용할 때 주의해야 할 점을 조사하고 있어요."}, config)
graph.invoke({"messages": "또 어떤 주제에 대해 공부하면 좋을까요?"}, config)
final_response = graph.invoke({"messages": "제가 뭘 공부하고 있나요?"}, config)

if final_response.get('summary'):
    print(f"요약: {final_response['summary']}\n")
final_response["messages"][-1].pretty_print()

요약: 당신은 AI를 공부하는 학생이며, AI를 사용할 때 주의해야 할 점을 조사하고 있습니다. 이에 대해 윤리적 고려, 데이터 프라이버시, 투명성, 신뢰성, 보안 문제, 책임, 사회적 영향과 같은 요소를 고려해야 한다고 설명받았습니다. 추가로, AI 분야에서 공부할 만한 다양한 주제들에 대한 추천도 받았습니다. 추천된 주제들에는 머신러닝 알고리즘, 딥러닝, 자연어 처리, 컴퓨터 비전, 강화학습, AI 윤리, 데이터 사이언스, AI 응용 분야 등이 포함되었습니다.

================================== Ai Message ==================================

당신은 AI(인공지능)를 공부하는 학생으로 보입니다. AI를 사용하거나 개발할 때 주의해야 할 다양한 윤리적, 기술적 문제에 대해 조사하고 있으며, AI 분야의 다양한 주제에 대해 관심을 가지고 있음을 이전 대화에서 알 수 있습니다. AI의 윤리적 고려, 데이터 프라이버시, 투명성, 신뢰성, 보안 문제, 책임, 사회적 영향 등에 대해 배우고 있으며, 머신러닝, 딥러닝, 자연어 처리, 컴퓨터 비전, 강화학습 같은 기술적 주제도 다루고 있다고 알고 있습니다. 추가적인 정보나 도움이 필요하다면 언제든지 물어보세요!
